In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, VBox, interactive_output

# 1. Create test signal N = 32
np.random.seed(42)
sample_values = np.round(np.random.uniform(1, 10, 32) + 1j * np.random.uniform(0, 5, 32), 2)
N = len(sample_values)
num_stages = int(np.log2(N))

# 2. Bit-reversal indexing
def bit_reverse_indices(n_pts):
    bits = int(np.log2(n_pts))
    rev = np.zeros(n_pts, dtype=int)
    for i in range(n_pts):
        r = 0
        for j in range(bits):
            if (i & (1 << j)) != 0:
                r |= (1 << (bits - 1 - j))
        rev[i] = r
    return rev

rev_idx = bit_reverse_indices(N)

# 3. Stage simulation and storage of numerical logs
stage_data = []
stage_math = []

curr = sample_values[rev_idx].astype(complex)
stage_data.append(curr.copy())
stage_math.append("Stage 0: Initial Bit-Reversed State\n- No synthesis operations yet.\n- Isolated single-point samples (Length L = 1).")

for s in range(1, num_stages + 1):
    L = 1 << s        # Sub-block length (2, 4, 8, 16, 32)
    half_L = L >> 1
    nxt = np.zeros_like(curr)
    
    group_name = "Pairs (L=2)" if L==2 else ("Quadruplets (L=4)" if L==4 else ("Octuplets (L=8)" if L==8 else ("16-tuples (L=16)" if L==16 else "32-tuples (L=32)")))
    
    math_log = f"Stage {s} Numeric Simulation ({group_name}, L = {L}):\n"
    math_log += f"Showing explicit numerical values & twiddle factors:\n"
    math_log += "-" * 55 + "\n"
    
    for i in range(0, N, L):
        math_log += f"[Block start index {i}]\n"
        for j in range(half_L):
            angle = -2 * np.pi * j / L
            w = np.exp(1j * angle)
            u = curr[i + j]
            v = curr[i + j + half_L] * w
            
            out_u = u + v
            out_v = u - v
            
            nxt[i + j]             = out_u
            nxt[i + j + half_L]     = out_v
            
            w_str = f"{w.real:.2f}{'+' if w.imag>=0 else ''}{w.imag:.2f}j" if abs(w.imag)>1e-4 else f"{w.real:.2f}"
            math_log += f"  j={j} (k={j}, W={w_str}):\n"
            math_log += f"    X_{s}[{i+j}]   = {u.real:.2f}+{u.imag:.2f}j + ({w_str})*({v.real:.2f}+{v.imag:.2f}j) = {out_u.real:.2f}+{out_u.imag:.2f}j\n"
            math_log += f"    X_{s}[{i+j+half_L}] = {u.real:.2f}+{u.imag:.2f}j - ({w_str})*({v.real:.2f}+{v.imag:.2f}j) = {out_v.real:.2f}+{out_v.imag:.2f}j\n"
            
    curr = nxt
    stage_data.append(curr.copy())
    stage_math.append(math_log)

color_list = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def plot_interactive_synthesis_32_magnitude(stage):
    # Clear current figure to prevent overlapping plots in Jupyter
    plt.close('all')
    fig = plt.figure(figsize=(18, 9.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1], height_ratios=[1, 1], hspace=0.32, wspace=0.15)
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax_math = fig.add_subplot(gs[:, 1])
    
    n_pts = np.arange(N)
    
    # --- TOP PLOT: Input state (Magnitude) ---
    L_in = 1 << max(0, stage - 1)
    mags_in = np.abs(stage_data[max(0, stage - 1)])
    ax1.set_title(f"Stage {stage} Input (Sub-block Length L = {L_in}) - Bit-Reversed Flow", fontsize=11, fontweight='bold', pad=8)
    
    max_val_in = np.max(mags_in) if np.max(mags_in) > 0 else 1.0
    ax1.set_ylim(0, max_val_in * 1.35)
    
    for i in range(N):
        block_id = i // max(1, L_in)
        c = color_list[block_id % len(color_list)]
        ax1.stem([i], [mags_in[i]], linefmt=c, markerfmt='o', basefmt='k-')
        # Exactly ~0.5 cm fixed offset (approx 14 points) above the marker using annotate
        ax1.annotate(f"{mags_in[i]:.1f}", (i, mags_in[i]),
                     textcoords="offset points", xytext=(0, 14),
                     ha='center', fontsize=6.5, fontweight='bold', color=c)
        
    ax1.set_ylabel("Magnitude |X[k]|", fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.set_xticks(n_pts)
    ax1.set_xticklabels([str(i) for i in n_pts], fontsize=8)

    # --- BOTTOM PLOT: Output state (Magnitude) ---
    L_out = 1 << stage
    mags_out = np.abs(stage_data[stage])
    group_label = f"Length L={L_out}"
    ax2.set_title(f"Stage {stage} Output: Synthesized {group_label}", fontsize=11, fontweight='bold', pad=8)
    
    max_val_out = np.max(mags_out) if np.max(mags_out) > 0 else 1.0
    ax2.set_ylim(0, max_val_out * 1.35)
    
    for i in range(N):
        block_id = i // max(1, L_out)
        c = color_list[block_id % len(color_list)]
        ax2.stem([i], [mags_out[i]], linefmt=c, markerfmt='s', basefmt='k-')
        # Exactly ~0.5 cm fixed offset (approx 14 points) above the marker using annotate
        ax2.annotate(f"{mags_out[i]:.1f}", (i, mags_out[i]),
                     textcoords="offset points", xytext=(0, 14),
                     ha='center', fontsize=6.5, fontweight='bold', color=c)
        
    ax2.set_xlabel("Index ($k$)", fontsize=10)
    ax2.set_ylabel("Magnitude |X[k]|", fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.set_xticks(n_pts)
    ax2.set_xticklabels([str(i) for i in n_pts], fontsize=8)

    # --- RIGHT PANEL: Numeric Simulation Logs (Full Height) ---
    ax_math.axis('off')
    math_text = stage_math[stage]
    ax_math.text(0.02, 0.98, math_text, fontsize=7.5, family='monospace', va='top', ha='left',
                 bbox=dict(boxstyle='round,pad=1', facecolor='whitesmoke', edgecolor='lightgray', alpha=0.95))

    plt.show()

# Robust widget binding using interactive_output to ensure smooth updates
stage_slider = IntSlider(value=1, min=1, max=num_stages, step=1, description='Stage:')
out = interactive_output(plot_interactive_synthesis_32_magnitude, {'stage': stage_slider})

display(VBox([stage_slider, out]))